In [4]:
# ============================================================
# 2_feature_engineering.ipynb
# Feature Engineering + Label Creation
# ============================================================

# Install Spark (Colab)
!pip install pyspark==3.5.1 -q

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -----------------------------
# Start Spark
# -----------------------------
spark = (
    SparkSession.builder
    .appName("TLC_Feature_Engineering")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# -----------------------------
# Load Cleaned Dataset
# -----------------------------
df = spark.read.parquet("/content/drive/MyDrive/Colab Notebooks/tlc_cleaned_parquet")

print("Dataset loaded.")

# -----------------------------
# Additional Feature Engineering
# -----------------------------

# Pickup hour
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

# Day of week
df = df.withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))

# Weekend flag
df = df.withColumn(
    "is_weekend",
    F.when(F.col("pickup_dayofweek").isin([1, 7]), 1).otherwise(0)
)

# Rush hour flag
df = df.withColumn(
    "is_rush_hour",
    F.when(
        (F.col("pickup_hour").between(7, 9)) |
        (F.col("pickup_hour").between(16, 19)),
        1
    ).otherwise(0)
)

# Average speed (mph)
df = df.withColumn(
    "avg_speed_mph",
    F.when(
        F.col("trip_seconds") > 0,
        F.col("trip_distance") / (F.col("trip_seconds") / 3600.0)
    ).otherwise(0)
)

# -----------------------------
# Create Label (Binary Classification)
# -----------------------------
df = df.withColumn(
    "label",
    (F.col("tip_amount") > 0).cast("int")
)

# -----------------------------
# Select Final Columns for ML
# -----------------------------
final_columns = [
    # "PU_Borough", # This column does not exist in the DataFrame. It might need to be created from PULocationID.
    "PULocationID", # Using PULocationID which is available in the DataFrame.
    "payment_type",
    "trip_distance",
    "trip_seconds",
    "fare_amount",
    "total_amount",
    "passenger_count",
    "pickup_hour",
    "pickup_dayofweek",
    "is_weekend",
    "is_rush_hour",
    "avg_speed_mph",
    "label",
    "pickup_date"
]

df_final = df.select(final_columns)

# -----------------------------
# Save Engineered Dataset
# -----------------------------
output_path = "data/processed/tlc_engineered_parquet"

df_final.repartition(100, "pickup_date") \
    .write \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet(output_path)

print("Feature engineered dataset saved to:", output_path)

# -----------------------------
# Stop Spark
# -----------------------------
spark.stop()

print("2_feature_engineering.ipynb completed successfully.")

Dataset loaded.
Feature engineered dataset saved to: data/processed/tlc_engineered_parquet
2_feature_engineering.ipynb completed successfully.
